# Complete Model Training Notebook for Dissertation

This notebook trains **ALL models** in the codebase and outputs comprehensive metrics and visualizations.

## Models Covered (22 Total)
1. **Baseline Models (5)**: LSTM, GRU, BiLSTM, Attention LSTM, Transformer
2. **PINN Models (6)**: Baseline PINN, GBM, OU, Black-Scholes, GBM+OU, Global
3. **Advanced PINN (3)**: StackedPINN, ResidualPINN, SpectralPINN
4. **Volatility Models (6)**: Vol-LSTM, Vol-GRU, Vol-Transformer, Vol-PINN, Heston-PINN, Stacked-Vol-PINN
5. **Dual-Phase PDE PINNs (2)**: Burgers PINN, Dual-Phase Burgers PINN

## Setup Instructions
1. Use GPU runtime: Runtime -> Change runtime type -> GPU
2. Clone or upload the repo to Colab
3. Run all cells in order


In [ ]:
#@title 1. Environment Setup
import os
import sys
import subprocess
from pathlib import Path
from getpass import getpass

PROJECT_ROOT = os.environ.get('PROJECT_ROOT', '/content/Dissertaion-Project')

if 'google.colab' in sys.modules:
    REPO_OWNER = 'must1f' # Assuming this is the owner of the private repo
    REPO_NAME = 'Dissertaion-Project'

    # Check if PROJECT_ROOT exists and is not a valid git repo, then remove it.
    # This handles cases of partial or failed clones.
    if os.path.exists(PROJECT_ROOT) and not Path(PROJECT_ROOT).joinpath('.git').is_dir():
        print(f"Removing incomplete repository at {PROJECT_ROOT}...")
        subprocess.run(['rm', '-rf', PROJECT_ROOT], check=True)

    # Clone only if the directory does not exist after potential cleanup
    if not os.path.exists(PROJECT_ROOT):
        github_pat = os.environ.get('GITHUB_PAT')
        if not github_pat:
            print("Please enter your GitHub Personal Access Token (PAT).")
            print("You can generate one here: https://github.com/settings/tokens")
            github_pat = getpass("GitHub PAT: ")
            os.environ['GITHUB_PAT'] = github_pat

        REPO_URL = f'https://{github_pat}@github.com/{REPO_OWNER}/{REPO_NAME}'
        print(f'Cloning private repository to {PROJECT_ROOT}...')
        subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)
    else:
        print(f'Repository already cloned to {PROJECT_ROOT}. Skipping clone step.')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print('GPU Info:')
    print(result.stdout if result.stdout else 'No GPU detected')
except FileNotFoundError:
    print('nvidia-smi not found - running on CPU')

print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'numpy', 'pandas', 'matplotlib', 'seaborn',
                'scikit-learn', 'tqdm', 'yfinance', 'python-dotenv', 'pydantic'], check=True)

req_file = Path(PROJECT_ROOT) / 'backend' / 'requirements.txt'
if req_file.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)], check=False)

print('Setup complete!')

In [ ]:
#@title 2. Import Core Modules
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

try:
    from src.models.model_registry import ModelRegistry
    from src.training.trainer import Trainer
    from src.data.fetcher import DataFetcher
    from src.data.preprocessor import DataPreprocessor
    from src.data.dataset import PhysicsAwareDataset, collate_fn_with_metadata
    from src.utils.config import get_config, get_research_config, get_dynamic_date_range
    from src.utils.reproducibility import set_seed, get_device
    from src.evaluation.metrics import calculate_metrics, calculate_financial_metrics
    from src.evaluation.financial_metrics import compute_strategy_returns, FinancialMetrics
    from src.constants import TRANSACTION_COST, RISK_FREE_RATE, TRADING_DAYS_PER_YEAR
    from src.models.dp_pinn import create_burgers_pinn
    from src.training.dp_pinn_trainer import DPPINNTrainer, TrainingConfig
    from src.utils.sampling import generate_burgers_training_data
    from src.evaluation.pde_evaluator import create_burgers_evaluator
    HAS_SRC = True
    print('Successfully imported src modules - using REAL neural network training')
except ImportError as e:
    HAS_SRC = False
    print(f'Failed to import src modules: {e}')
    raise

# Import volatility trainer
try:
    from src.training.volatility_trainer import VolatilityDataPreparer, VolatilityTrainer
    HAS_VOL_TRAINER = True
    print('Volatility trainer imported successfully')
except ImportError as e:
    HAS_VOL_TRAINER = False
    print(f'Volatility trainer not available: {e}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

set_seed(42)
print('Random seed set to 42')


In [ ]:
#@title 2b. Reduce Training Output (Epoch-only Progress)
class _SilentTQDM:
    def __init__(self, iterable, **kwargs):
        self.iterable = iterable
    def __iter__(self):
        for item in self.iterable:
            yield item
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        return False
    def set_postfix(self, *args, **kwargs):
        return None
    def update(self, *args, **kwargs):
        return None

def use_epoch_only_progress():
    import src.training.trainer as trainer_module
    trainer_module.tqdm = _SilentTQDM
    print('Batch-level progress bars disabled (epoch-only progress).')

use_epoch_only_progress()


In [ ]:
#@title 3. Define Model Groups (ALL 22 MODELS)
project_root = Path(PROJECT_ROOT)
registry = ModelRegistry(project_root)
research_cfg = get_research_config()

# Baseline Models (5)
BASELINE_MODELS = ['lstm', 'gru', 'bilstm', 'attention_lstm', 'transformer']

# PINN Models (6)
PINN_MODELS = ['baseline_pinn', 'gbm', 'ou', 'black_scholes', 'gbm_ou', 'global']

# Advanced PINN Models (3)
ADVANCED_PINN_MODELS = ['stacked', 'residual', 'spectral_pinn']

# Volatility Models (6)
VOLATILITY_MODELS = ['vol_lstm', 'vol_gru', 'vol_transformer', 'vol_pinn', 'heston_pinn', 'stacked_vol_pinn']
# Dual-Phase PINN (Burgers PDE) Models (2)
DP_MODELS = ['burgers_pinn', 'dual_phase_pinn']

# All price prediction models
PRICE_MODELS = BASELINE_MODELS + PINN_MODELS + ADVANCED_PINN_MODELS

# Total model count (price + vol + dp)
TOTAL_MODELS = len(PRICE_MODELS) + len(VOLATILITY_MODELS) + len(DP_MODELS)

print(f'ModelRegistry contains {len(registry.models)} model definitions (including aliases)')
print(f'\n=== MODELS TO TRAIN ({TOTAL_MODELS} total) ===')
print(f'\nBaseline models ({len(BASELINE_MODELS)}): {BASELINE_MODELS}')
print(f'PINN models ({len(PINN_MODELS)}): {PINN_MODELS}')
print(f'Advanced PINN ({len(ADVANCED_PINN_MODELS)}): {ADVANCED_PINN_MODELS}')
print(f'Volatility models ({len(VOLATILITY_MODELS)}): {VOLATILITY_MODELS}')
print(f'DP PINN models ({len(DP_MODELS)}): {DP_MODELS}')
print(f'\nResearch config: epochs={research_cfg.epochs}, batch_size={research_cfg.batch_size}')


In [ ]:
#@title 4. Prepare Price Prediction Training Data
def prepare_training_data(ticker='^GSPC', use_multi_ticker=True, force_refresh=False):
    config = get_config()
    research_cfg = get_research_config()

    start_date, end_date = get_dynamic_date_range(10)
    config.data.start_date = start_date
    config.data.end_date = end_date

    fetcher = DataFetcher()
    preprocessor = DataPreprocessor()

    base_tickers = [ticker]
    if use_multi_ticker:
        base_tickers.extend(config.data.tickers[:10])
    tickers = sorted(set(base_tickers))
    print(f'Fetching S&P 500 (^GSPC) data from {config.data.start_date} to {config.data.end_date}')
    if use_multi_ticker:
        extra_count = max(0, len(tickers) - 1)
        preview = tickers[1:6] if extra_count else []
        print(f'Including {extra_count} constituent tickers: {preview}...')

    df = fetcher.fetch_and_store(
        tickers=tickers,
        start_date=config.data.start_date,
        end_date=config.data.end_date,
        force_refresh=force_refresh
    )

    if df.empty:
        raise ValueError('No data fetched!')
    if '^GSPC' not in df['ticker'].unique():
        raise ValueError('S&P 500 (^GSPC) data missing from fetched dataset')

    df = preprocessor.process_and_store(df)

    # Match research-grade pipeline feature set from scripts/train_models.py
    research_features = [
        'close', 'volume',
        'log_return', 'simple_return',
        'rolling_volatility_5', 'rolling_volatility_20',
        'momentum_5', 'momentum_20',
        'rsi_14', 'macd', 'macd_signal',
        'bollinger_upper', 'bollinger_lower', 'atr_14',
    ]
    feature_cols = [col for col in research_features if col in df.columns]

    required = ['close', 'log_return']
    missing = [f for f in required if f not in feature_cols]
    if missing:
        raise ValueError(f'Missing required features: {missing}')

    print(f'Using {len(feature_cols)} features')

    train_df, val_df, test_df = preprocessor.split_temporal(df)

    for col in feature_cols:
        train_df[col] = train_df[col].astype(np.float64)
        val_df[col] = val_df[col].astype(np.float64)
        test_df[col] = test_df[col].astype(np.float64)

    train_df_norm, scalers = preprocessor.normalize_features(train_df, feature_cols, method='standard')

    val_df_norm = val_df.copy()
    test_df_norm = test_df.copy()

    for ticker_name in val_df['ticker'].unique():
        if ticker_name in scalers:
            val_mask = val_df_norm['ticker'] == ticker_name
            val_df_norm.loc[val_mask, feature_cols] = scalers[ticker_name].transform(
                val_df_norm.loc[val_mask, feature_cols])

    for ticker_name in test_df['ticker'].unique():
        if ticker_name in scalers:
            test_mask = test_df_norm['ticker'] == ticker_name
            test_df_norm.loc[test_mask, feature_cols] = scalers[ticker_name].transform(
                test_df_norm.loc[test_mask, feature_cols])

    seq_length = research_cfg.sequence_length
    target_col = 'close'  # Match prior Python pipeline

    X_train, y_train, tickers_train = preprocessor.create_sequences(
        train_df_norm, feature_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)

    X_val, y_val, tickers_val = preprocessor.create_sequences(
        val_df_norm, feature_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)

    X_test, y_test, tickers_test = preprocessor.create_sequences(
        test_df_norm, feature_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)

    physics_cols = ['close', 'log_return', 'rolling_volatility_20']
    missing_physics = [c for c in physics_cols if c not in df.columns]
    if missing_physics:
        raise ValueError(f'Missing physics metadata columns: {missing_physics}')

    P_train, _, _ = preprocessor.create_sequences(
        train_df, physics_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)
    P_val, _, _ = preprocessor.create_sequences(
        val_df, physics_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)
    P_test, _, _ = preprocessor.create_sequences(
        test_df, physics_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)

    train_ds = PhysicsAwareDataset(
        X_train, y_train, tickers_train,
        prices=P_train[:, :, 0], returns=P_train[:, :, 1],
        volatilities=P_train[:, :, 2])

    val_ds = PhysicsAwareDataset(
        X_val, y_val, tickers_val,
        prices=P_val[:, :, 0], returns=P_val[:, :, 1],
        volatilities=P_val[:, :, 2])

    test_ds = PhysicsAwareDataset(
        X_test, y_test, tickers_test,
        prices=P_test[:, :, 0], returns=P_test[:, :, 1],
        volatilities=P_test[:, :, 2])

    input_dim = X_train.shape[2]

    print(f'Dataset sizes: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}')
    print(f'Input dimension: {input_dim}')
    print(f'Sequence length: {seq_length}')

    return train_ds, val_ds, test_ds, input_dim, scalers

train_ds, val_ds, test_ds, input_dim, scalers = prepare_training_data()


In [ ]:
#@title 5. Prepare Volatility Training Data
def prepare_volatility_data():
    """Prepare data specifically for volatility forecasting models."""
    if not HAS_VOL_TRAINER:
        print('Volatility trainer not available - skipping volatility data prep')
        return None
    
    config = get_config()
    fetcher = DataFetcher()
    preprocessor = DataPreprocessor()

    start_date, end_date = get_dynamic_date_range(10)
    config.data.start_date = start_date
    config.data.end_date = end_date
    
    print(f'Fetching data for volatility models (^GSPC) from {start_date} to {end_date}')
    vol_df = fetcher.fetch_and_store(
        tickers=['^GSPC'],
        start_date=start_date,
        end_date=end_date,
        force_refresh=False
    )
    vol_df = preprocessor.process_and_store(vol_df)
    single = vol_df[vol_df['ticker'] == '^GSPC'].sort_values('time').set_index('time')
    
    # Prepare volatility features
    preparer = VolatilityDataPreparer(seq_length=40, horizon=5)
    features = preparer.compute_volatility_features(single)
    
    # Drop NaN rows
    features = features.dropna()
    aligned_returns = single.loc[features.index, 'log_return']
    
    dataset = preparer.prepare(
        features.values, 
        aligned_returns.values, 
        dates=features.index.values
    )
    
    print(f'Volatility dataset: Train={dataset.X_train.shape[0]}, Val={dataset.X_val.shape[0]}')
    print(f'Volatility features: {dataset.X_train.shape[2]}')
    
    return dataset

vol_dataset = prepare_volatility_data()

In [ ]:
#@title 6. Training Helper Functions
train_loader = DataLoader(train_ds, batch_size=research_cfg.batch_size,
                          shuffle=True, collate_fn=collate_fn_with_metadata)
val_loader = DataLoader(val_ds, batch_size=research_cfg.batch_size,
                        shuffle=False, collate_fn=collate_fn_with_metadata)
test_loader = DataLoader(test_ds, batch_size=research_cfg.batch_size,
                         shuffle=False, collate_fn=collate_fn_with_metadata)

results_dir = project_root / 'results' / 'colab_runs'
results_dir.mkdir(parents=True, exist_ok=True)


def _to_python(obj):
    if isinstance(obj, dict):
        return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_python(v) for v in obj]
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def _extract_close_stats(scalers):
    if not scalers or not isinstance(scalers, dict):
        return None, None
    if len(scalers) != 1:
        return None, None

    sc = next(iter(scalers.values()))
    mean = getattr(sc, 'mean_', None)
    std = getattr(sc, 'scale_', None)
    if mean is None or std is None:
        return None, None

    mean_val = float(np.squeeze(mean[0]) if hasattr(mean, 'shape') and len(mean.shape) > 0 else float(mean))
    std_val = float(np.squeeze(std[0]) if hasattr(std, 'shape') and len(std.shape) > 0 else float(std))
    return mean_val, std_val


def compute_all_evaluation_metrics(predictions, targets, model_name, scalers=None):
    metrics = {}

    price_mean, price_std = _extract_close_stats(scalers)

    pred_metrics = calculate_metrics(
        targets,
        predictions,
        prefix='',
        price_mean=price_mean,
        price_std=price_std,
    )

    metrics.update({
        'rmse': pred_metrics.get('rmse'),
        'mae': pred_metrics.get('mae'),
        'mape': pred_metrics.get('mape'),
        'mse': pred_metrics.get('mse'),
        'r2': pred_metrics.get('r2'),
        'directional_accuracy': pred_metrics.get('directional_accuracy'),
    })

    try:
        strategy_returns = compute_strategy_returns(
            predictions=predictions,
            actual_prices=targets,
            are_returns=False,
            transaction_cost=TRANSACTION_COST,
            price_mean=price_mean,
            price_std=price_std,
        )

        fin_metrics = calculate_financial_metrics(
            returns=strategy_returns,
            risk_free_rate=RISK_FREE_RATE,
            periods_per_year=TRADING_DAYS_PER_YEAR,
            prefix='',
        )

        metrics.update({
            'sharpe_ratio': fin_metrics.get('sharpe_ratio'),
            'sortino_ratio': fin_metrics.get('sortino_ratio'),
            'max_drawdown': fin_metrics.get('max_drawdown'),
            'calmar_ratio': fin_metrics.get('calmar_ratio'),
            'total_return': fin_metrics.get('total_return'),
            'volatility': fin_metrics.get('volatility'),
            'win_rate': fin_metrics.get('win_rate'),
        })

        if len(strategy_returns) > 0:
            metrics['annualized_return'] = FinancialMetrics.annualized_return(
                strategy_returns, periods_per_year=TRADING_DAYS_PER_YEAR
            ) * 100

    except Exception as e:
        print(f'Warning: Financial metrics failed for {model_name}: {e}')

    return metrics


def evaluate_model(model, loader, model_name, scalers=None):
    model.eval()
    predictions, targets = [], []

    with torch.no_grad():
        for batch in loader:
            sequences, target, metadata = batch
            sequences = sequences.to(device)
            target = target.to(device)

            output = model(sequences)
            if isinstance(output, tuple):
                output = output[0]

            predictions.append(output.cpu().numpy().squeeze())
            targets.append(target.cpu().numpy().squeeze())

    predictions = np.concatenate(predictions)
    targets = np.concatenate(targets)

    metrics = compute_all_evaluation_metrics(
        predictions=predictions,
        targets=targets,
        model_name=model_name,
        scalers=scalers,
    )

    return predictions, targets, metrics


def train_price_model(model_key, epochs=None, scalers=None, enable_physics=None):
    print(f'\n{"="*60}')
    print(f'Training: {model_key}')
    print(f'{"="*60}')

    epochs = epochs or research_cfg.epochs

    model = registry.create_model(
        model_type=model_key,
        input_dim=input_dim,
        hidden_dim=research_cfg.hidden_dim,
        num_layers=research_cfg.num_layers,
        dropout=research_cfg.dropout,
    )

    if model is None:
        print(f'Failed to create model: {model_key}')
        return None

    is_pinn = hasattr(model, 'compute_loss')
    if enable_physics is None:
        enable_physics = is_pinn

    print(f'Model: {model.__class__.__name__}')
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
    print(f'Physics-enabled training: {bool(enable_physics and is_pinn)}')

    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        research_mode=True,
    )

    history = {
        'epochs': [],
        'train_loss': [],
        'val_loss': [],
        'data_loss': [],
        'physics_loss': [],
        'learning_rates': [],
    }

    best_val_loss = float('inf')

    pbar = tqdm(range(1, epochs + 1), desc=model_key)
    for epoch in pbar:
        train_loss, train_metrics = trainer.train_epoch(enable_physics=bool(enable_physics and is_pinn))
        val_loss, _ = trainer.validate_epoch(enable_physics=bool(enable_physics and is_pinn))

        history['epochs'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['data_loss'].append(train_metrics.get('train_data_loss'))
        history['physics_loss'].append(train_metrics.get('train_physics_loss'))
        history['learning_rates'].append(trainer.optimizer.param_groups[0]['lr'])

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            trainer.save_checkpoint(epoch=epoch, val_loss=val_loss,
                                    is_best=True, model_name=model_key)

        pbar.set_postfix({'train': f'{train_loss:.4f}', 'val': f'{val_loss:.4f}'})

    predictions, targets, metrics = evaluate_model(
        trainer.model,
        test_loader,
        model_key,
        scalers=scalers,
    )

    print('Test Metrics:')
    print(f'  RMSE: {metrics.get("rmse", float("nan")):.4f}')
    print(f'  MAE: {metrics.get("mae", float("nan")):.4f}')
    print(f'  R²: {metrics.get("r2", float("nan")):.4f}')
    print(f'  Directional Accuracy: {metrics.get("directional_accuracy", float("nan")):.2f}%')
    print(f'  Sharpe Ratio: {metrics.get("sharpe_ratio", float("nan")):.3f}')
    print(f'  Sortino Ratio: {metrics.get("sortino_ratio", float("nan")):.3f}')
    print(f'  Max Drawdown: {metrics.get("max_drawdown", float("nan")):.2f}')

    payload = {
        'model': model_key,
        'history': history,
        'test_metrics': metrics,
        'metrics': metrics,
        'training_completed': True,
        'best_val_loss': best_val_loss,
        'epochs_trained': epochs,
    }

    with open(results_dir / f'{model_key}_results.json', 'w') as f:
        json.dump(_to_python(payload), f, indent=2)

    return {
        'history': history,
        'metrics': metrics,
        'predictions': predictions,
        'targets': targets,
    }


def train_volatility_model(model_key, dataset, epochs=30):
    if dataset is None or not HAS_VOL_TRAINER:
        print(f'Skipping {model_key} - volatility trainer not available')
        return None

    print(f'\n{"="*60}')
    print(f'Training Volatility Model: {model_key}')
    print(f'{"="*60}')

    model = registry.create_model(
        model_key,
        input_dim=dataset.X_train.shape[2],
        hidden_dim=64,
        num_layers=2,
        dropout=0.1,
    )

    if model is None:
        print(f'Failed to create model: {model_key}')
        return None

    print(f'Model: {model.__class__.__name__}')
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

    trainer = VolatilityTrainer(
        model,
        learning_rate=1e-3,
        batch_size=64,
        max_epochs=epochs,
        patience=10,
        device=device,
        checkpoint_dir=results_dir,
    )

    enable_physics = 'pinn' in model_key or 'heston' in model_key
    history = trainer.fit(dataset, enable_physics=enable_physics, verbose=True)

    val_losses = history.get('val_loss', [])
    best_epoch = int(np.argmin(val_losses)) if val_losses else None

    metrics = {
        'final_train_loss': history.get('train_loss', [np.nan])[-1] if history.get('train_loss') else np.nan,
        'final_val_loss': history.get('val_loss', [np.nan])[-1] if history.get('val_loss') else np.nan,
        'best_val_loss': np.min(history.get('val_loss', [np.nan])) if history.get('val_loss') else np.nan,
        'final_val_mse': history.get('val_mse', [np.nan])[-1] if history.get('val_mse') else np.nan,
        'final_val_mae': history.get('val_mae', [np.nan])[-1] if history.get('val_mae') else np.nan,
        'final_val_qlike': history.get('val_qlike', [np.nan])[-1] if history.get('val_qlike') else np.nan,
        'final_val_r2': history.get('val_r2', [np.nan])[-1] if history.get('val_r2') else np.nan,
        'epochs_trained': len(history.get('train_loss', [])),
    }

    if best_epoch is not None:
        metrics.update({
            'best_epoch': best_epoch + 1,
            'best_val_mse': history.get('val_mse', [np.nan])[best_epoch] if history.get('val_mse') else np.nan,
            'best_val_mae': history.get('val_mae', [np.nan])[best_epoch] if history.get('val_mae') else np.nan,
            'best_val_qlike': history.get('val_qlike', [np.nan])[best_epoch] if history.get('val_qlike') else np.nan,
            'best_val_r2': history.get('val_r2', [np.nan])[best_epoch] if history.get('val_r2') else np.nan,
        })

    if dataset.X_test is not None and dataset.y_test is not None:
        try:
            test_metrics = trainer.evaluate(dataset.X_test, dataset.y_test)
            for key, value in test_metrics.items():
                if isinstance(value, (int, float, np.integer, np.floating)):
                    metrics[f'test_{key}'] = float(value)
        except Exception as e:
            print(f'Warning: Test metrics failed for {model_key}: {e}')

    print('Volatility Metrics:')
    print(f'  Best Val Loss: {metrics.get("best_val_loss", float("nan")):.6f}')
    print(f'  Best Val QLIKE: {metrics.get("best_val_qlike", float("nan")):.6f}')
    print(f'  Best Val R²: {metrics.get("best_val_r2", float("nan")):.4f}')

    payload = {
        'model': model_key,
        'history': history,
        'metrics': metrics,
        'training_completed': True,
    }

    with open(results_dir / f'{model_key}_results.json', 'w') as f:
        json.dump(_to_python(payload), f, indent=2)

    return {
        'history': history,
        'metrics': metrics,
    }

print('Training functions defined!')


In [ ]:
#@title 7. Train Baseline Models (5)
baseline_results = {}

for model_key in BASELINE_MODELS:
    try:
        result = train_price_model(model_key, scalers=scalers, enable_physics=False)
        if result:
            baseline_results[model_key] = result
    except Exception as e:
        print(f'Error training {model_key}: {e}')
        import traceback
        traceback.print_exc()

print(f'\nCompleted training {len(baseline_results)}/{len(BASELINE_MODELS)} baseline models')


In [ ]:
#@title 8. Train PINN Models (6)
pinn_results = {}

for model_key in PINN_MODELS:
    try:
        result = train_price_model(model_key, scalers=scalers, enable_physics=True)
        if result:
            pinn_results[model_key] = result
    except Exception as e:
        print(f'Error training {model_key}: {e}')
        import traceback
        traceback.print_exc()

print(f'\nCompleted training {len(pinn_results)}/{len(PINN_MODELS)} PINN models')


In [ ]:
#@title 9. Train Advanced PINN Models (3) - Including Spectral
advanced_results = {}

for model_key in ADVANCED_PINN_MODELS:
    try:
        result = train_price_model(model_key, scalers=scalers, enable_physics=True)
        if result:
            advanced_results[model_key] = result
    except Exception as e:
        print(f'Error training {model_key}: {e}')
        import traceback
        traceback.print_exc()

print(f'\nCompleted training {len(advanced_results)}/{len(ADVANCED_PINN_MODELS)} advanced PINN models')


In [ ]:
#@title 10. Train Volatility Models (6)
volatility_results = {}

for model_key in VOLATILITY_MODELS:
    try:
        result = train_volatility_model(model_key, vol_dataset, epochs=30)
        if result:
            volatility_results[model_key] = result
    except Exception as e:
        print(f'Error training {model_key}: {e}')
        import traceback
        traceback.print_exc()

print(f'\nCompleted training {len(volatility_results)}/{len(VOLATILITY_MODELS)} volatility models')

In [ ]:
#@title 11. Train Dual-Phase (Burgers) PINN Models
dp_results = {}
dp_checkpoints_dir = results_dir / 'dp_pinn'
dp_checkpoints_dir.mkdir(parents=True, exist_ok=True)

dp_config = TrainingConfig(device=str(device))
burgers_data = generate_burgers_training_data(device=device)
burgers_evaluator = create_burgers_evaluator(device=device)

# Standard Burgers PINN
trainer_burg = DPPINNTrainer(create_burgers_pinn(model_type='standard').to(device), dp_config, device=device)
trainer_burg.train_standard_pinn(burgers_data, eval_fn=burgers_evaluator.evaluate_all)
trainer_burg.save_checkpoint(dp_checkpoints_dir / 'burgers_pinn.ckpt', extra_info={'model_key': 'burgers_pinn'})
burg_metrics = burgers_evaluator.evaluate_all(trainer_burg.model)
dp_results['burgers_pinn'] = {
    'metrics': burg_metrics.to_dict(),
    'history': trainer_burg.history.to_dict(),
}

# Dual-phase Burgers PINN
trainer_dp = DPPINNTrainer(create_burgers_pinn(model_type='dual_phase').to(device), dp_config, device=device)
phase1_history = trainer_dp.train_phase1(burgers_data, eval_fn=burgers_evaluator.evaluate_all)
phase1_history_dict = phase1_history.to_dict()
phase2_history = trainer_dp.train_phase2(burgers_data, eval_fn=burgers_evaluator.evaluate_all)
phase2_history_dict = phase2_history.to_dict()
trainer_dp.save_checkpoint(dp_checkpoints_dir / 'dual_phase_pinn.ckpt', extra_info={'model_key': 'dual_phase_pinn'})
dp_metrics = burgers_evaluator.evaluate_all(trainer_dp.model)
dp_results['dual_phase_pinn'] = {
    'metrics': dp_metrics.to_dict(),
    'phase1_history': phase1_history_dict,
    'phase2_history': phase2_history_dict,
}

# Persist DP results
with open(dp_checkpoints_dir / 'dp_pinn_results.json', 'w') as f:
    json.dump(_to_python(dp_results), f, indent=2)

print('\n=== DP PINN RESULTS ===')
for name, res in dp_results.items():
    m = res['metrics']
    print(f"{name}: L2_rel={m.get('relative_l2_error', float('nan')):.6f}, max_err={m.get('max_error', float('nan')):.6f}")


In [ ]:
#@title 12. Plot DP PINN Training Curves and Metrics
if dp_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Standard Burgers PINN losses
    std_hist = dp_results.get('burgers_pinn', {}).get('history', {})
    if std_hist:
        axes[0].plot(std_hist.get('iterations', []), std_hist.get('losses', []), label='Total Loss')
        axes[0].plot(std_hist.get('iterations', []), std_hist.get('pde_losses', []), label='PDE Loss', alpha=0.8)
        axes[0].set_title('Burgers PINN Loss')
        axes[0].set_xlabel('Iteration')
        axes[0].set_ylabel('Loss')
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

    # Dual-phase losses (phase 2 shown; phase1 captured in history)
    dp_hist = dp_results.get('dual_phase_pinn', {}).get('phase2_history', {})
    if dp_hist:
        axes[1].plot(dp_hist.get('iterations', []), dp_hist.get('losses', []), label='Total Loss')
        axes[1].plot(dp_hist.get('iterations', []), dp_hist.get('pde_losses', []), label='PDE Loss', alpha=0.8)
        axes[1].plot(dp_hist.get('iterations', []), dp_hist.get('intermediate_losses', []), label='Intermediate', alpha=0.8)
        axes[1].set_title('Dual-Phase PINN Loss (Phase 2)')
        axes[1].set_xlabel('Iteration')
        axes[1].set_ylabel('Loss')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

    plt.tight_layout()
    plt.savefig(results_dir / 'dp_pinn_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    # DP metrics summary
    dp_summary_rows = []
    for model_key, res in dp_results.items():
        m = res['metrics']
        dp_summary_rows.append({
            'Model': model_key,
            'Relative L2': m.get('relative_l2_error'),
            'Max Error': m.get('max_error'),
            'Mean Error': m.get('mean_error'),
            'Mean Residual': m.get('mean_residual'),
        })

    dp_summary_df = pd.DataFrame(dp_summary_rows)
    display(dp_summary_df.round(6))
    dp_summary_df.to_csv(results_dir / 'dp_pinn_summary.csv', index=False)
    print(f"DP PINN summary saved to {results_dir / 'dp_pinn_summary.csv'}")
else:
    print('No DP PINN results available')


In [ ]:
#@title 13. Combine All Results
price_results = {**baseline_results, **pinn_results, **advanced_results}
all_results = {**price_results, **volatility_results, **dp_results}

print(f'=== TRAINING SUMMARY ===')
print(f'Total models trained: {len(all_results)}')
print(f'  - Baseline: {len(baseline_results)}')
print(f'  - PINN: {len(pinn_results)}')
print(f'  - Advanced PINN: {len(advanced_results)}')
print(f'  - Volatility: {len(volatility_results)}')
print(f'  - DP PINN: {len(dp_results)}')
print(f'\nAll models: {list(all_results.keys())}')

In [ ]:
#@title 13b. Metrics Coverage Audit (All Models)\ndef audit_metrics():\n    audit = {'price': {}, 'volatility': {}, 'dp': {}}\n    required_price = ['rmse', 'mae', 'mape', 'r2', 'directional_accuracy', 'sharpe_ratio', 'sortino_ratio', 'max_drawdown']\n    required_vol = ['best_val_loss', 'best_val_qlike', 'best_val_r2', 'final_val_mse', 'final_val_mae']\n    required_dp = ['relative_l2_error', 'max_error', 'mean_error', 'mean_residual']\n    \n    for name, res in price_results.items():\n        metrics = res.get('metrics', {})\n        missing = [m for m in required_price if metrics.get(m) is None]\n        audit['price'][name] = {'missing': missing, 'ok': len(missing) == 0}\n    \n    for name, res in volatility_results.items():\n        metrics = res.get('metrics', {})\n        missing = [m for m in required_vol if metrics.get(m) is None]\n        audit['volatility'][name] = {'missing': missing, 'ok': len(missing) == 0}\n    \n    for name, res in dp_results.items():\n        metrics = res.get('metrics', {})\n        missing = [m for m in required_dp if metrics.get(m) is None]\n        audit['dp'][name] = {'missing': missing, 'ok': len(missing) == 0}\n    \n    audit['artifacts'] = {\n        'price_summary_csv': (results_dir / 'price_model_summary.csv').exists(),\n        'vol_summary_csv': (results_dir / 'volatility_model_summary.csv').exists(),\n        'dp_summary_csv': (results_dir / 'dp_pinn_summary.csv').exists(),\n        'price_training_png': (results_dir / 'price_training_curves.png').exists(),\n        'vol_training_png': (results_dir / 'volatility_training_curves.png').exists(),\n        'metrics_comparison_png': (results_dir / 'metrics_comparison.png').exists(),\n        'vol_metrics_png': (results_dir / 'volatility_metrics_comparison.png').exists(),\n        'dp_training_png': (results_dir / 'dp_pinn_training_curves.png').exists(),\n        'pred_vs_actual_png': (results_dir / 'predictions_vs_actual_all_price_models.png').exists(),\n    }\n    \n    with open(results_dir / 'metrics_audit.json', 'w') as f:\n        json.dump(_to_python(audit), f, indent=2)\n    \n    print('Metrics audit saved to', results_dir / 'metrics_audit.json')\n    for category, models in audit.items():\n        if category == 'artifacts':\n            continue\n        for name, entry in models.items():\n            status = 'OK' if entry['ok'] else f"Missing: {entry['missing']}"\n            print(f"{category.upper()} - {name}: {status}")\n    return audit\n\naudit_results = audit_metrics()\n

In [ ]:
#@title 14. Generate Price Models Metrics Summary
summary_rows = []

for model_key, result in price_results.items():
    metrics = result['metrics']
    
    if model_key in BASELINE_MODELS:
        model_type = 'Baseline'
    elif model_key in PINN_MODELS:
        model_type = 'PINN'
    else:
        model_type = 'Advanced PINN'
    
    summary_rows.append({
        'Model': model_key,
        'Type': model_type,
        'RMSE': metrics.get('rmse'),
        'MAE': metrics.get('mae'),
        'MAPE': metrics.get('mape'),
        'R2': metrics.get('r2'),
        'Dir. Acc. (%)': metrics.get('directional_accuracy'),
        'Sharpe': metrics.get('sharpe_ratio'),
        'Sortino': metrics.get('sortino_ratio'),
        'Max DD (%)': metrics.get('max_drawdown', 0) * 100 if metrics.get('max_drawdown') else None
    })

price_summary_df = pd.DataFrame(summary_rows)
price_summary_df = price_summary_df.sort_values('RMSE')

print('\n' + '='*80)
print('PRICE PREDICTION MODEL PERFORMANCE SUMMARY')
print('='*80)
display(price_summary_df.round(4))

price_summary_df.to_csv(results_dir / 'price_model_summary.csv', index=False)
print(f'Summary saved to {results_dir / "price_model_summary.csv"}')

In [ ]:
#@title 15. Generate Volatility Models Metrics Summary
vol_summary_rows = []

for model_key, result in volatility_results.items():
    metrics = result['metrics']

    is_physics = 'pinn' in model_key or 'heston' in model_key

    vol_summary_rows.append({
        'Model': model_key,
        'Type': 'Physics-Informed' if is_physics else 'Baseline',
        'Best Val Loss': metrics.get('best_val_loss'),
        'Best Val QLIKE': metrics.get('best_val_qlike'),
        'Best Val R2': metrics.get('best_val_r2'),
        'Final Val MSE': metrics.get('final_val_mse'),
        'Final Val MAE': metrics.get('final_val_mae'),
        'Epochs Trained': metrics.get('epochs_trained'),
    })

vol_summary_df = pd.DataFrame(vol_summary_rows)

if not vol_summary_df.empty:
    sort_col = 'Best Val QLIKE' if 'Best Val QLIKE' in vol_summary_df.columns else 'Best Val Loss'
    vol_summary_df = vol_summary_df.sort_values(sort_col)

print('\n' + '='*80)
print('VOLATILITY MODEL PERFORMANCE SUMMARY')
print('='*80)
display(vol_summary_df.round(6))

vol_summary_df.to_csv(results_dir / 'volatility_model_summary.csv', index=False)
print(f'Summary saved to {results_dir / "volatility_model_summary.csv"}')


In [ ]:
#@title 16. Plot Training Curves - Price Models
n_models = len(price_results)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten() if n_models > 1 else [axes]

for idx, (model_key, result) in enumerate(price_results.items()):
    ax = axes[idx]
    history = result['history']
    
    ax.plot(history['epochs'], history['train_loss'], label='Train', linewidth=2)
    ax.plot(history['epochs'], history['val_loss'], label='Val', linewidth=2, linestyle='--')
    
    ax.set_title(f'{model_key}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

for idx in range(n_models, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Training Curves - Price Prediction Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(results_dir / 'price_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Training curves saved to {results_dir / "price_training_curves.png"}')

In [ ]:
#@title 17. Plot Training Curves - Volatility Models
if volatility_results:
    n_vol = len(volatility_results)
    n_cols_vol = min(3, n_vol)
    n_rows_vol = (n_vol + n_cols_vol - 1) // n_cols_vol
    
    fig, axes = plt.subplots(n_rows_vol, n_cols_vol, figsize=(15, 4*n_rows_vol))
    if n_vol > 1:
        axes = axes.flatten()
    else:
        axes = [axes]
    
    for idx, (model_key, result) in enumerate(volatility_results.items()):
        ax = axes[idx]
        history = result['history']
        
        if 'train_loss' in history and 'val_loss' in history:
            epochs = range(1, len(history['train_loss']) + 1)
            ax.plot(epochs, history['train_loss'], label='Train', linewidth=2)
            ax.plot(epochs, history['val_loss'], label='Val', linewidth=2, linestyle='--')
        
        ax.set_title(f'{model_key}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    for idx in range(n_vol, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Training Curves - Volatility Models', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(results_dir / 'volatility_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'Volatility training curves saved to {results_dir / "volatility_training_curves.png"}')
else:
    print('No volatility models trained')

In [ ]:
#@title 18. Plot Metrics Comparison - All Price Models
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = []
for model in price_summary_df['Model']:
    if model in BASELINE_MODELS:
        colors.append('#2E86AB')
    elif model in PINN_MODELS:
        colors.append('#A23B72')
    else:
        colors.append('#F18F01')

ax1 = axes[0, 0]
ax1.barh(price_summary_df['Model'], price_summary_df['RMSE'], color=colors)
ax1.set_xlabel('RMSE (lower is better)')
ax1.set_title('RMSE by Model')
ax1.invert_yaxis()

ax2 = axes[0, 1]
ax2.barh(price_summary_df['Model'], price_summary_df['Dir. Acc. (%)'], color=colors)
ax2.set_xlabel('Directional Accuracy %')
ax2.set_title('Directional Accuracy by Model')
ax2.invert_yaxis()
ax2.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='Random (50%)')

ax3 = axes[1, 0]
sharpe_sorted = price_summary_df.sort_values('Sharpe', ascending=True)
colors_sharpe = []
for model in sharpe_sorted['Model']:
    if model in BASELINE_MODELS:
        colors_sharpe.append('#2E86AB')
    elif model in PINN_MODELS:
        colors_sharpe.append('#A23B72')
    else:
        colors_sharpe.append('#F18F01')

ax3.barh(sharpe_sorted['Model'], sharpe_sorted['Sharpe'], color=colors_sharpe)
ax3.set_xlabel('Sharpe Ratio (higher is better)')
ax3.set_title('Sharpe Ratio by Model')
ax3.invert_yaxis()
ax3.axvline(x=0, color='red', linestyle='--', alpha=0.5)

ax4 = axes[1, 1]
type_metrics = price_summary_df.groupby('Type').agg({
    'RMSE': 'mean',
    'Dir. Acc. (%)': 'mean',
    'Sharpe': 'mean'
}).round(4)

x = np.arange(len(type_metrics))
width = 0.25

rmse_norm = type_metrics['RMSE'] / type_metrics['RMSE'].max()
da_norm = type_metrics['Dir. Acc. (%)'] / 100
sharpe_norm = (type_metrics['Sharpe'] - type_metrics['Sharpe'].min()) / \
              (type_metrics['Sharpe'].max() - type_metrics['Sharpe'].min() + 1e-6)

ax4.bar(x - width, rmse_norm, width, label='RMSE (norm)', color='#2E86AB')
ax4.bar(x, da_norm, width, label='Dir. Acc. (norm)', color='#A23B72')
ax4.bar(x + width, sharpe_norm, width, label='Sharpe (norm)', color='#F18F01')
ax4.set_xticks(x)
ax4.set_xticklabels(type_metrics.index)
ax4.set_ylabel('Normalized Score')
ax4.set_title('Average Metrics by Model Type')
ax4.legend()

plt.tight_layout()
plt.savefig(results_dir / 'metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Metrics comparison saved to {results_dir / "metrics_comparison.png"}')

In [ ]:
#@title 19. Plot Metrics Comparison - All Volatility Models
if volatility_results:
    vol_metrics_rows = []

    for model_key, result in volatility_results.items():
        m = result['metrics']
        vol_metrics_rows.append({
            'Model': model_key,
            'Best Val QLIKE': m.get('best_val_qlike', np.nan),
            'Best Val R2': m.get('best_val_r2', np.nan),
            'Final Val Loss': m.get('final_val_loss', np.nan),
            'Type': 'Physics-Informed' if ('pinn' in model_key or 'heston' in model_key) else 'Baseline',
        })

    vol_metrics_df = pd.DataFrame(vol_metrics_rows).sort_values('Best Val QLIKE', ascending=True)

    colors = ['#A23B72' if t == 'Physics-Informed' else '#2E86AB' for t in vol_metrics_df['Type']]

    fig, axes = plt.subplots(1, 3, figsize=(18, max(5, 0.45 * len(vol_metrics_df) + 2)))

    axes[0].barh(vol_metrics_df['Model'], vol_metrics_df['Best Val QLIKE'], color=colors)
    axes[0].set_title('Best Validation QLIKE (lower better)')
    axes[0].set_xlabel('QLIKE')
    axes[0].invert_yaxis()

    axes[1].barh(vol_metrics_df['Model'], vol_metrics_df['Best Val R2'], color=colors)
    axes[1].set_title('Best Validation R² (higher better)')
    axes[1].set_xlabel('R²')
    axes[1].invert_yaxis()
    axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)

    axes[2].barh(vol_metrics_df['Model'], vol_metrics_df['Final Val Loss'], color=colors)
    axes[2].set_title('Final Validation Loss (lower better)')
    axes[2].set_xlabel('Loss')
    axes[2].invert_yaxis()

    plt.tight_layout()
    plt.savefig(results_dir / 'volatility_metrics_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Volatility metrics comparison saved to {results_dir / "volatility_metrics_comparison.png"}')
else:
    print('No volatility models trained')


In [ ]:
#@title 20. Plot Predictions vs Actuals (All Price Models)
if price_results:
    n_models = len(price_results)
    n_cols = 3
    n_rows = (n_models + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3.8 * n_rows))
    axes = np.array(axes).flatten() if n_models > 1 else [axes]

    for idx, (model_key, result) in enumerate(price_results.items()):
        ax = axes[idx]

        preds = np.asarray(result['predictions']).flatten()
        targs = np.asarray(result['targets']).flatten()
        n = min(200, len(preds), len(targs))

        ax.plot(targs[:n], label='Actual', alpha=0.8, linewidth=1.6)
        ax.plot(preds[:n], label='Predicted', alpha=0.8, linewidth=1.2)

        rmse = result['metrics'].get('rmse', np.nan)
        ax.set_title(f'{model_key} | RMSE={rmse:.4f}', fontsize=10)
        ax.set_xlabel('Sample')
        ax.set_ylabel('Close (normalized)')
        ax.grid(True, alpha=0.25)

        if idx == 0:
            ax.legend(fontsize=8)

    for idx in range(n_models, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Predictions vs Actuals - All Price Models', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(results_dir / 'predictions_vs_actual_all_price_models.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Prediction comparison saved to {results_dir / "predictions_vs_actual_all_price_models.png"}')
else:
    print('No price models trained')


In [ ]:
#@title 21. Final Comprehensive Report
print('='*80)
print('FINAL TRAINING REPORT - ALL MODELS')
print('='*80)
print(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Device: {device}')
print(f'Research Config: epochs={research_cfg.epochs}, batch_size={research_cfg.batch_size}')

print(f'\n=== TRAINING SUMMARY ===')
print(f'Total models trained: {len(all_results)}')
print(f'  - Baseline: {len(baseline_results)}/{len(BASELINE_MODELS)}')
print(f'  - PINN: {len(pinn_results)}/{len(PINN_MODELS)}')
print(f'  - Advanced PINN (incl. Spectral): {len(advanced_results)}/{len(ADVANCED_PINN_MODELS)}')
print(f'  - Volatility: {len(volatility_results)}/{len(VOLATILITY_MODELS)}')

print('\n' + '-'*40)
print('BEST PRICE PREDICTION MODELS')
print('-'*40)

if len(price_summary_df) > 0:
    best_rmse = price_summary_df.loc[price_summary_df['RMSE'].idxmin()]
    print(f'Best RMSE: {best_rmse["Model"]} ({best_rmse["RMSE"]:.4f})')

    best_da = price_summary_df.loc[price_summary_df['Dir. Acc. (%)'].idxmax()]
    print(f'Best Directional Accuracy: {best_da["Model"]} ({best_da["Dir. Acc. (%)"]:.2f}%)')

    best_sharpe = price_summary_df.loc[price_summary_df['Sharpe'].idxmax()]
    print(f'Best Sharpe Ratio: {best_sharpe["Model"]} ({best_sharpe["Sharpe"]:.3f})')

print('\n' + '-'*40)
print('BEST VOLATILITY MODELS')
print('-'*40)

if len(vol_summary_df) > 0:
    best_vol = vol_summary_df.loc[vol_summary_df['Best Val QLIKE'].idxmin()]
    print(f'Best Val QLIKE: {best_vol["Model"]} ({best_vol["Best Val QLIKE"]:.6f})')

print('\n' + '-'*40)
print('PINN vs BASELINE COMPARISON (Price Models)')
print('-'*40)

if len(price_summary_df) > 0:
    baseline_avg = price_summary_df[price_summary_df['Type'] == 'Baseline'].mean(numeric_only=True)
    pinn_avg = price_summary_df[price_summary_df['Type'] == 'PINN'].mean(numeric_only=True)

    if not baseline_avg.empty and not pinn_avg.empty:
        print(f'Average RMSE:')
        print(f'  Baseline: {baseline_avg["RMSE"]:.4f}')
        print(f'  PINN: {pinn_avg["RMSE"]:.4f}')
        improvement = (1 - pinn_avg["RMSE"]/baseline_avg["RMSE"])*100
        print(f'  Improvement: {improvement:.2f}%')

        print(f'\nAverage Directional Accuracy:')
        print(f'  Baseline: {baseline_avg["Dir. Acc. (%)"]:.2f}%')
        print(f'  PINN: {pinn_avg["Dir. Acc. (%)"]:.2f}%')

        print(f'\nAverage Sharpe Ratio:')
        print(f'  Baseline: {baseline_avg["Sharpe"]:.3f}')
        print(f'  PINN: {pinn_avg["Sharpe"]:.3f}')

print('\n' + '='*80)
print(f'Results saved to: {results_dir}')
print('Files generated:')
print('  - price_model_summary.csv')
print('  - volatility_model_summary.csv')
print('  - price_training_curves.png')
print('  - volatility_training_curves.png')
print('  - metrics_comparison.png')
print('  - volatility_metrics_comparison.png')
print('  - predictions_vs_actual_all_price_models.png')
print('  - dp_pinn_training_curves.png')
print('  - dp_pinn_summary.csv')
print('  - {model}_results.json for each model (price/vol)')
print('  - dp_pinn/dp_pinn_results.json (Burgers PINNs)')
print('='*80)
